# Car Price Prediction — Milestone 4: Further Improvements

**Dataset:** [Car Price Prediction Challenge — Kaggle](https://www.kaggle.com/datasets/deepcontractor/car-price-prediction-challenge)

### Before running
Upload the four CSV files from M2: `X_train_prepared.csv`, `X_test_prepared.csv`, `y_train_prepared.csv`, `y_test_prepared.csv`

---
| Direction | Type | Method |
|---|---|---|
| **Direction 1** | Advanced model training | Gradient Boosting Regressor + 5-fold GridSearchCV |
| **Direction 2** | Target engineering | log(1+Price) transformation on training target |

Structure: 1. Imports · 2. Load data · 3. M3 reference · 4. Direction 1 · 5. Direction 2 · 6. Comparison · 7. Discussion · 8. Conclusion

## 1. Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'figure.dpi': 120})
PAL = sns.color_palette('Set2')

def evaluate(name, y_true, y_pred):
    """Compute RMSE, MAE and R2 for a set of predictions."""
    return {
        'Model':     name,
        'Test RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'Test MAE':  mean_absolute_error(y_true, y_pred),
        'Test R2':   r2_score(y_true, y_pred),
    }
print('RANDOM_SEED =', RANDOM_SEED)

## 2. Load M2 Prepared Data

In [ ]:
X_train = pd.read_csv('X_train_prepared.csv')
X_test  = pd.read_csv('X_test_prepared.csv')
y_train = pd.read_csv('y_train_prepared.csv').squeeze()
y_test  = pd.read_csv('y_test_prepared.csv').squeeze()
print(f'X_train {X_train.shape}  X_test {X_test.shape}')
print(f'Price skewness: {y_train.skew():.2f}')

## 3. M3 Reference Results
Best M3 model: **Ridge (alpha=10)** — RMSE $10,700 | R² 0.2843. Both M4 directions are benchmarked against this.

In [ ]:
ridge_m3 = Ridge(alpha=10, random_state=RANDOM_SEED)
ridge_m3.fit(X_train, y_train)
ridge_pred = ridge_m3.predict(X_test)
res_ridge  = evaluate('Ridge (M3)', y_test, ridge_pred)

rf_m3 = RandomForestRegressor(n_estimators=100, max_depth=10,
                               min_samples_split=2, random_state=RANDOM_SEED, n_jobs=-1)
rf_m3.fit(X_train, y_train)
rf_pred = rf_m3.predict(X_test)
res_rf  = evaluate('Random Forest (M3)', y_test, rf_pred)

print(f"Ridge  RMSE ${res_ridge['Test RMSE']:,.0f}  R2 {res_ridge['Test R2']:.4f}  <- M3 benchmark")
print(f"RF     RMSE ${res_rf['Test RMSE']:,.0f}  R2 {res_rf['Test R2']:.4f}")

# Residual fan shape — motivates both M4 directions
resid_ridge = y_test.values - ridge_pred
fig, axes   = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(ridge_pred, resid_ridge, alpha=0.2, s=8, color=PAL[0])
axes[0].axhline(0, color='red', lw=1.5, linestyle='--')
axes[0].set_xlabel('Predicted Price ($)'); axes[0].set_ylabel('Residual ($)')
axes[0].set_title('M3 Ridge — Residuals vs Predicted\n(fan shape = heteroscedasticity)')
axes[0].annotate('Variance grows\nwith price', xy=(28000,35000), xytext=(4000,52000),
    fontsize=9, color='darkred', arrowprops=dict(arrowstyle='->', color='darkred', lw=1.5))
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].hist(resid_ridge, bins=60, color=PAL[0], edgecolor='none', alpha=0.85)
axes[1].axvline(0, color='red', lw=1.5, linestyle='--')
axes[1].set_xlabel('Residual ($)'); axes[1].set_ylabel('Count')
axes[1].set_title('M3 Ridge — Residual Distribution\n(right tail: premium cars underpredicted)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.suptitle('Figure 1 — M3 Ridge Residual Analysis: Motivation for Both M4 Directions', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

## 4. Direction 1 — Gradient Boosting Regressor

### What it does
GBM builds decision trees **sequentially**. Each new tree is trained to correct the residual errors of all previous trees by fitting the negative gradient of the loss function. Unlike Random Forest (parallel independent trees), GBM focuses every new tree on the current mistakes. Shallow trees (max_depth = 3–5) with many boosting rounds can model smooth non-linear curves precisely.

Key hyperparameters:
- **n_estimators** — number of boosting rounds
- **max_depth** — depth of each tree (shallow = regularised)
- **learning_rate** — shrinkage applied to each tree's contribution

### Why this direction?
- **Not used in M3** — required condition satisfied
- Fundamentally different learning strategy: boosting vs bagging vs linear regression
- The M3 fan-shaped residuals show systematic underprediction at high prices; GBM's sequential correction may specifically address these errors
- Shallow trees with many rounds model smooth non-linear price curves more precisely than deep RF trees

### Hyperparameter tuning — 5-fold GridSearchCV
Grid: n_estimators ∈ {100, 200}, max_depth ∈ {3, 5}, learning_rate ∈ {0.05, 0.1} → 8 combinations × 5 folds = 40 fits

In [ ]:
param_grid = {
    'n_estimators':  [100, 200],
    'max_depth':     [3, 5],
    'learning_rate': [0.05, 0.1],
}

gbm_cv = GridSearchCV(
    GradientBoostingRegressor(random_state=RANDOM_SEED),
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    verbose=1
)
gbm_cv.fit(X_train, y_train)
best_gbm = gbm_cv.best_estimator_

# Unpack best hyperparameters for use in Direction 2
best_n_est  = gbm_cv.best_params_['n_estimators']
best_depth  = gbm_cv.best_params_['max_depth']
best_lr     = gbm_cv.best_params_['learning_rate']

print(f'Best params  : {gbm_cv.best_params_}')
print(f'Best CV RMSE : ${-gbm_cv.best_score_:,.0f}')

In [ ]:
# CV RMSE heatmap (averaged over n_estimators)
cv_df = pd.DataFrame(gbm_cv.cv_results_)
cv_df['mean_cv_rmse'] = -cv_df['mean_test_score']
heat = cv_df.groupby(['param_learning_rate','param_max_depth'])['mean_cv_rmse'].mean().unstack()
annot = heat.map(lambda v: f'${v:,.0f}' if not pd.isna(v) else 'N/A')

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(heat, annot=annot, fmt='', cmap='YlOrRd_r', ax=ax,
            linewidths=0.6, cbar_kws={'label': 'Mean CV RMSE ($)'})
ax.set_xlabel('max_depth'); ax.set_ylabel('learning_rate')
ax.set_title(f'Figure 2 — GBM 5-Fold CV RMSE: learning_rate x max_depth\n'
             f'Best: {gbm_cv.best_params_}  |  Best CV RMSE = ${-gbm_cv.best_score_:,.0f}',
             fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# GBM learning curve — staged RMSE vs number of boosting rounds
gbm_pred = best_gbm.predict(X_test)
res_gbm  = evaluate('GBM (Dir. 1)', y_test, gbm_pred)
d1_delta = res_gbm['Test RMSE'] - res_ridge['Test RMSE']
sign = '+' if d1_delta > 0 else ''
print(f"Test RMSE : ${res_gbm['Test RMSE']:,.0f}")
print(f"Test MAE  : ${res_gbm['Test MAE']:,.0f}")
print(f"Test R2   : {res_gbm['Test R2']:.4f}")
print(f"Delta vs M3 Ridge: {sign}${abs(d1_delta):,.0f}")

train_errors, test_errors = [], []
for pred in best_gbm.staged_predict(X_train):
    train_errors.append(np.sqrt(mean_squared_error(y_train, pred)))
for pred in best_gbm.staged_predict(X_test):
    test_errors.append(np.sqrt(mean_squared_error(y_test, pred)))

best_idx  = int(np.argmin(test_errors))
best_test = test_errors[best_idx]
n_rounds  = list(range(1, len(train_errors) + 1))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_rounds, train_errors, color=PAL[0], lw=2, label='Training RMSE')
ax.plot(n_rounds, test_errors,  color=PAL[1], lw=2, label='Test RMSE')
ax.axvline(best_idx + 1, color='green', lw=1.5, linestyle='--',
           label=f'Best test RMSE at n={best_idx+1} (${best_test:,.0f})')
ax.scatter([best_idx + 1], [best_test], color='green', s=80, zorder=5)
ax.set_xlabel('Number of Boosting Rounds (n_estimators)')
ax.set_ylabel('RMSE ($)')
ax.set_title(f'Figure 3 — GBM Learning Curve: Training vs Test RMSE per Boosting Round\n'
             f'(best model: depth={best_depth}, lr={best_lr})', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(fontsize=10); sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# GBM predictions and residuals
resid_gbm = y_test.values - gbm_pred
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, gbm_pred, alpha=0.25, s=8, color=PAL[4])
axes[0].plot([y_test.min(), y_test.max()],[y_test.min(), y_test.max()],
             'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($)'); axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f"GBM — Test R2 = {res_gbm['Test R2']:.3f}  |  RMSE = ${res_gbm['Test RMSE']:,.0f}")
axes[0].legend()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].scatter(gbm_pred, resid_gbm, alpha=0.2, s=8, color=PAL[4])
axes[1].axhline(0, color='red', lw=1.5, linestyle='--')
axes[1].set_xlabel('Predicted Price ($)'); axes[1].set_ylabel('Residual ($)')
axes[1].set_title('GBM — Residuals vs Predicted\n(fan shape persists: non-linearity not the root cause)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.suptitle('Figure 4 — Gradient Boosting: Predictions & Residuals', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

## 5. Direction 2 — Log(1+Price) Target Transformation

### Motivation
The M3 residual fan shape is a symptom of **heteroscedasticity** — the target price spans two orders of magnitude, so proportional prediction errors look large in dollar terms for premium cars.

log(1+Price) compresses large values. The raw price skewness drops from 1.83 to -0.28 after transformation. If a model overpredicts consistently by a percentage factor, that becomes a constant additive error in log-space, which RMSE can minimise uniformly.

### Method
1. `y_train_log = log(1 + y_train)` — transform training target only (no leakage)
2. Train Ridge, RF, and GBM on `y_train_log` with the same M2 features
3. Back-transform: `y_pred = exp(y_pred_log) − 1`
4. Evaluate in original dollar scale — directly comparable to M3

In [ ]:
y_train_log = np.log1p(y_train)
print(f'Price skewness:     {y_train.skew():.2f}')
print(f'Log-price skewness: {y_train_log.skew():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_train, bins=60, color=PAL[3], edgecolor='none', alpha=0.85)
axes[0].set_xlabel('Price ($)'); axes[0].set_ylabel('Count')
axes[0].set_title(f'Original Price  (skewness = {y_train.skew():.2f})')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].hist(y_train_log, bins=60, color=PAL[2], edgecolor='none', alpha=0.85)
axes[1].set_xlabel('log(1 + Price)'); axes[1].set_ylabel('Count')
axes[1].set_title(f'Log-Transformed Price  (skewness = {y_train_log.skew():.2f})')
plt.suptitle('Figure 5 — Log(1+Price) Transformation: Before and After', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

# Train all three families on log-price
ridge_log = Ridge(alpha=10, random_state=RANDOM_SEED)
ridge_log.fit(X_train, y_train_log)
ridge_log_pred = np.expm1(ridge_log.predict(X_test))
res_ridge_log  = evaluate('Ridge + log(y)', y_test, ridge_log_pred)

rf_log = RandomForestRegressor(n_estimators=100, max_depth=10,
                                min_samples_split=2, random_state=RANDOM_SEED, n_jobs=-1)
rf_log.fit(X_train, y_train_log)
rf_log_pred = np.expm1(rf_log.predict(X_test))
res_rf_log  = evaluate('RF + log(y)', y_test, rf_log_pred)

# Use best GBM config found in Direction 1
gbm_log = GradientBoostingRegressor(n_estimators=best_n_est, max_depth=best_depth,
                                     learning_rate=best_lr, random_state=RANDOM_SEED)
gbm_log.fit(X_train, y_train_log)
gbm_log_pred = np.expm1(gbm_log.predict(X_test))
res_gbm_log  = evaluate('GBM + log(y)', y_test, gbm_log_pred)

ref = res_ridge['Test RMSE']
print('Direction 2 results (dollar scale):')
for r in [res_ridge_log, res_rf_log, res_gbm_log]:
    d = r['Test RMSE'] - ref
    sign = '+' if d > 0 else ''
    print(f"  {r['Model']:28s}  RMSE ${r['Test RMSE']:,.0f}  delta {sign}${abs(d):,.0f}")

In [ ]:
resid_rl = y_test.values - ridge_log_pred
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, ridge_log_pred, alpha=0.25, s=8, color=PAL[1])
axes[0].plot([y_test.min(), y_test.max()],[y_test.min(), y_test.max()],
             'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($)'); axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f"Ridge + log(y) — R2 = {res_ridge_log['Test R2']:.3f}  |  "
                  f"RMSE = ${res_ridge_log['Test RMSE']:,.0f}")
axes[0].legend()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].scatter(ridge_log_pred, resid_rl, alpha=0.2, s=8, color=PAL[1])
axes[1].axhline(0, color='red', lw=1.5, linestyle='--')
axes[1].set_xlabel('Predicted Price ($)'); axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Ridge + log(y) — Residuals\n(fan shape reduced, but RMSE is higher)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.suptitle('Figure 6 — Ridge + Log-Price Target: Predictions & Residuals', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

## 6. Cross-Milestone Comparison

In [ ]:
all_res = pd.DataFrame([res_ridge, res_rf, res_gbm,
                        res_ridge_log, res_rf_log, res_gbm_log])
ref = res_ridge['Test RMSE']
print(f"  {'Model':<30}  {'RMSE':>9}  {'MAE':>9}  {'R2':>8}  {'Delta':>10}")
print('-' * 72)
for _, row in all_res.iterrows():
    d = row['Test RMSE'] - ref
    delta_s = f"+${d:,.0f}" if d > 0 else ("$0 (ref)" if abs(d)<1 else f"-${abs(d):,.0f}")
    mark = '  <- benchmark' if row['Model'] == 'Ridge (M3)' else ''
    print(f"  {row['Model']:<30}  ${row['Test RMSE']:>8,.0f}  ${row['Test MAE']:>8,.0f}"
          f"  {row['Test R2']:>8.4f}  {delta_s:>10}{mark}")

In [ ]:
order  = ['Ridge\n(M3)','RF\n(M3)','GBM\n(Dir.1)',
          'Ridge\n+log(y)','RF\n+log(y)','GBM\n+log(y)']
rmse_v = [res_ridge['Test RMSE'],     res_rf['Test RMSE'],     res_gbm['Test RMSE'],
          res_ridge_log['Test RMSE'], res_rf_log['Test RMSE'], res_gbm_log['Test RMSE']]
r2_v   = [res_ridge['Test R2'],       res_rf['Test R2'],       res_gbm['Test R2'],
          res_ridge_log['Test R2'],   res_rf_log['Test R2'],   res_gbm_log['Test R2']]
cols_b = [PAL[0], PAL[1], PAL[4], PAL[0], PAL[1], PAL[4]]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
bars = axes[0].bar(order, rmse_v, color=cols_b, edgecolor='white', width=0.65)
for j in range(3, 6): bars[j].set_hatch('//'); bars[j].set_alpha(0.7)
axes[0].axhline(res_ridge['Test RMSE'], color='green', lw=2, linestyle='--', label='M3 Ridge benchmark')
for bar, val in zip(bars, rmse_v):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+100, f'${val:,.0f}',
                 ha='center', va='bottom', fontsize=8.5, fontweight='bold')
axes[0].set_ylabel('Test RMSE ($)'); axes[0].set_title('RMSE — All Models', fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].legend(fontsize=9); axes[0].set_ylim(0, max(rmse_v)*1.20)
bars2 = axes[1].bar(order, r2_v, color=cols_b, edgecolor='white', width=0.65)
for j in range(3,6): bars2[j].set_hatch('//'); bars2[j].set_alpha(0.7)
axes[1].axhline(res_ridge['Test R2'], color='green', lw=2, linestyle='--', label='M3 Ridge benchmark')
for bar, val in zip(bars2, r2_v):
    axes[1].text(bar.get_x()+bar.get_width()/2, max(val,0)+0.006, f'{val:.3f}',
                 ha='center', va='bottom', fontsize=8.5, fontweight='bold')
axes[1].set_ylabel('Test R²'); axes[1].set_title('R² — All Models', fontweight='bold')
axes[1].legend(fontsize=9)
plt.suptitle('Figure 7 — Cross-Milestone Comparison\n'
             '(solid = M3/Dir.1  |  hatched = Dir.2 log-price  |  green = M3 Ridge benchmark)',
             fontweight='bold', fontsize=11)
sns.despine(); plt.tight_layout(); plt.show()

deltas = [v - res_ridge['Test RMSE'] for v in rmse_v]
dcols  = ['limegreen' if d<=0 else 'salmon' for d in deltas]
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(order, deltas, color=dcols, edgecolor='white', width=0.6)
for j in range(3,6): bars[j].set_hatch('//'); bars[j].set_alpha(0.8)
ax.axhline(0, color='black', lw=1.8)
for bar, val in zip(bars, deltas):
    ax.text(bar.get_x()+bar.get_width()/2, val+(12 if val>=0 else -80),
            f'+${val:,.0f}' if val>0 else f'${val:,.0f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('RMSE Delta vs M3 Ridge ($)')
ax.set_title('Figure 8 — RMSE Change Relative to M3 Ridge Benchmark', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:+,.0f}'))
sns.despine(); plt.tight_layout(); plt.show()

## 7. Discussion: Why Neither Direction Improved Over M3 Ridge

### Direction 1 — GBM approximately equals Ridge
GBM performs nearly identically to M3 Ridge (within $26 RMSE, <0.3%). This confirms the central M3 finding: **the M2 feature engineering pipeline had already linearised most of the price signal.**

The mechanism is **target encoding**: replacing Manufacturer and Model with their mean training prices converts a categorical price hierarchy into a continuous linear scale. The hand-crafted features (Car_Age, Km_Per_Year, Value_Score) further captured the remaining non-linear relationships explicitly. With so little residual non-linearity left, GBM's sequential trees find nothing more to exploit.

The learning curve (Figure 3) confirms this: test RMSE stabilises quickly and does not drop significantly beyond ~20 rounds — the model saturates early, which is consistent with an already-linear signal.

### Direction 2 — Log-price worsens all models
All log-price models perform worse (by +$276 to +$369 RMSE). The reason: **target encoding had already resolved the heteroscedasticity that log-transformation was designed to address.**

When manufacturers are encoded as mean prices (BMW → $45,000, Toyota → $12,000), the multiplicative price structure is absorbed into the feature representation. A linear model on these features already makes roughly proportional errors — exactly what log-transformation achieves. Applying both corrections double-counts: back-transformation via exp() then amplifies errors in the dense mid-price range, raising overall RMSE.

In [ ]:
# Why log hurts: residual spread by price tier
price_bins = pd.qcut(pd.Series(y_test), q=6, labels=False)
std_lin = [np.std(y_test.values[price_bins==b] - ridge_pred[price_bins==b])     for b in range(6)]
std_log = [np.std(y_test.values[price_bins==b] - ridge_log_pred[price_bins==b]) for b in range(6)]

bin_labels = ['Tier 1\n(lowest)','Tier 2','Tier 3','Tier 4','Tier 5','Tier 6\n(highest)']
x = np.arange(6); w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mfr_means = y_train.groupby(X_train['Manufacturer']).mean().sort_values()
axes[0].barh(mfr_means.index, mfr_means.values, color=PAL[0], edgecolor='white')
axes[0].set_xlabel('Target-Encoded Mean Price ($)', fontsize=10)
axes[0].set_title('Target Encoding Already Captures\nManufacturer Price Scale', fontsize=10, fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

axes[1].bar(x-w/2, std_lin, w, label='Ridge (linear target)', color=PAL[0], edgecolor='white')
axes[1].bar(x+w/2, std_log, w, label='Ridge + log(y)',        color=PAL[1], edgecolor='white', hatch='//')
axes[1].set_xticks(x); axes[1].set_xticklabels(bin_labels, fontsize=9)
axes[1].set_ylabel('Residual Std Dev ($)', fontsize=10)
axes[1].set_title('Residual Spread by Price Tier\n(log-price reduces high-end variance\nbut inflates mid-range errors)', fontsize=10, fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].legend(fontsize=9)
plt.suptitle('Figure 9 — Why Log-Price Did Not Help: Target Encoding Already Linearised the Scale',
             fontweight='bold', fontsize=11)
sns.despine(); plt.tight_layout(); plt.show()

## 8. Conclusion

In [ ]:
print('FINAL SUMMARY')
print('=' * 72)
rows = [
    ('Baseline (M3)',            12650,     -0.0003),
    ('Ridge  — M3 best',        10700,    0.2843),
    ('RF     — M3',             10764,   0.2758),
    ('GBM    — Direction 1',    10726,    0.2809),
    ('Ridge + log(y) — Dir. 2', 10976,   0.2469),
    ('RF    + log(y) — Dir. 2', 11048,  0.2370),
    ('GBM   + log(y) — Dir. 2', 11069,   0.2341),
]
ref = 10700
for name, rmse_v, r2v in rows:
    d = rmse_v - ref
    delta_s = f"+${d:,.0f}" if d > 0 else ("$0 (ref)" if abs(d)<1 else f"-${abs(d):,.0f}")
    mark = '  <- benchmark' if 'M3 best' in name else ''
    print(f"  {name:<32} ${rmse_v:>9,.0f}  R2 {r2v:.4f}  {delta_s:>12}{mark}")
print("""
Conclusions:
  1. GBM (Direction 1)  : Nearly identical to Ridge — target encoding
                          pre-linearised the price signal (confirmed by
                          the learning curve: RMSE saturates early).
  2. Log-price (Dir. 2) : All models worse — target encoding had already
                          resolved heteroscedasticity at the feature level.
  3. Both results validate the M2 pipeline.
  4. Further gains require new data (condition, service history, regional
     pricing), not different algorithms on the current feature set.""")